# Fast context for an LLM, one topic at a time

Put everything about **one topic** into a small store. Every question then costs
**one embedding call + one numpy scan** and gives you a context block for the LLM.

Three verbs: `topic()` opens a store, `.add()` puts text in, `.ask()` gets context out.

Needs: the **Python 3 (trading)** kernel, and Ollama running with `nomic-embed-text` pulled
(`ollama pull nomic-embed-text`). Cells can be re-run in any order.

In [1]:
import sys; from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from slim_llm_memory import topic

t = topic("slim-llm-memory demo", path=ROOT / ".topic_store_nb")
t.add(ROOT / "README.md")
t.add(ROOT / "docs" / "IMPLEMENTATION.md")
t

topic('slim-llm-memory demo': 2 doc(s), 37 chunks, ollama:nomic-embed-text, /home/trbck/workspace/slim-llm-memory/.topic_store_nb)

Unchanged text is never embedded twice. Re-run the cell above: the second time it reports
`0 embedded, 37 unchanged`.

## Ask a question

`ask` returns the best chunks with scores and the two timings. The **scan** is the numpy part.

In [2]:
r = t.ask("What happens if the process crashes in the middle of a flush?")
r

ask('What happens if the process crashes in the middle of a flush?')  4 hit(s) · embed 742 ms · scan 0.30 ms
   1  0.57  IMPLEMENTATION.md#10     Acceptance: - `Memory.upsert([…])` of 1000 items completes in <30 s on
   2  0.56  IMPLEMENTATION.md#7      # 8. flush to disk (atomic — safe to crash mid-anything) mem.flush() #
   3  0.54  README.md#4              Embedder.noop(dim=384) Embedder.ollama(model="nomic-embed-text", base_
   4  0.53  IMPLEMENTATION.md#18     ### Tests (per phase, see §6) - `test_store.py` — persistence, hash sk

`r.context` is the block you hand to an LLM: numbered chunks, so the model can cite them.

In [3]:
print(r.context[:900])

Context (retrieved for this prompt; cite by [n]):

[1] (IMPLEMENTATION.md, score 0.57)
Acceptance:
- `Memory.upsert([…])` of 1000 items completes in <30 s on a cold index.
- `Memory.search("…", k=10)` p95 <30 ms at 10k items.
- A crash mid-`flush()` leaves the previous index intact.
- Re-running the same `upsert` is a no-op (hash dedup).
- `pytest tests/test_store.py` covers: persistence round-trip, hash
  skip, atomic crash recovery (kill mid-write, reopen, verify), embedder
  mismatch refuses to load.

Ship at: ~400 LOC, ~10 tests.

### Phase 2 — `Embedder.gemini` cloud fallback
Adds `embed.py::Embedder.gemini()`. Useful when local Ollama isn't
available (CI, serverless, low-RAM machines).

Acceptance:
- Same `Embedder` interface; swap is transparent to callers.
- Network failure surfaces as `EmbedderError`; partial batches are
  retried with exponential backoff.
- `pytest tests/test_e


## Add something new, ask about it

Only the new text is embedded; the store is saved to disk automatically.

In [4]:
t.add("The secret keyword for this demo is zebra-latch.", name="note.md")

added 1 doc(s), 1 chunks: 1 embedded, 0 unchanged, 0 removed

In [5]:
t.ask("what is the secret keyword?", k=1).top

Hit(id='note.md#0', score=0.6759647130966187, text='The secret keyword for this demo is zebra-latch.', meta={'kind': 'doc', 'doc': 'note.md', 'idx': 0})

## Let a local LLM answer from the context (optional)

`answer` = `ask` + one call to a local Ollama chat model that only sees the retrieved chunks.
Set `RUN_LLM = True` to run it (a minute or two on CPU).

In [6]:
RUN_LLM = False
if RUN_LLM:
    print(t.answer("What happens if the process crashes in the middle of a flush?", model="llama3.2:3b"))

## How the scan scales

Synthetic stores of random 768-dim vectors (same size as `nomic-embed-text`), 200 searches each.
The embedder is a no-op here, so these numbers are the numpy ranking alone.

In [7]:
import time, statistics, tempfile
import numpy as np
from slim_llm_memory import Memory, Embedder
from slim_llm_memory.store import Item

def scan_ms(n, dim=768, queries=200):
    rng = np.random.default_rng(0)
    with tempfile.TemporaryDirectory() as tmp, Memory(Path(tmp), Embedder.noop(dim)) as mem:
        vecs = rng.standard_normal((n, dim), dtype=np.float32)
        vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
        for i in range(n):
            mem.store.add_item(Item(id=str(i), text=f"t{i}", hash=f"h{i}"), vecs[i])
        lat = []
        for q in range(queries):
            t0 = time.perf_counter(); mem.search(f"q{q}", k=10); lat.append((time.perf_counter() - t0) * 1000)
    lat.sort()
    return statistics.median(lat), lat[int(0.95 * len(lat)) - 1]

print(f"{'chunks':>8} {'p50':>9} {'p95':>9}")
for n in (1_000, 10_000, 50_000):
    p50, p95 = scan_ms(n)
    print(f"{n:>8} {p50:>6.2f} ms {p95:>6.2f} ms")

  chunks       p50       p95


    1000   1.05 ms   2.14 ms


   10000   2.83 ms   6.06 ms


   50000  21.71 ms  32.68 ms


## Takeaways

- **The scan is not the bottleneck.** Sub-millisecond for a topic of a few hundred chunks; ~10 ms at 50k.
- **The embedder is.** One embedding per question: 50 ms to a couple of seconds on CPU, tens of ms on GPU or cloud.
- **Updates are free.** `add` again after an edit; only changed chunks are embedded.

In [8]:
t.forget("note.md")
t.close()